# 1 · Three Faces of a Phase

*Phasor networks, from the ground up — notebook 1 of 7.*

Everything in a phasor network is built from one quantity: a **phase**. A single
phase value can be looked at three equivalent ways, and the whole framework comes
from moving fluidly between them:

| face | representation | lives in |
|------|----------------|----------|
| **a point on the unit circle** | the complex number `exp(iπθ)` | the complex plane |
| **a real angle** | a number `θ ∈ [-1, 1]` (units of π) | the real line |
| **a spike time** | *when* a neuron fires within one cycle | time |

This notebook introduces the three faces and the conversions between them. The
next notebook gives the phase a body — an **oscillator** that holds it in time.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using PhasorNetworks
using Plots

## Face 1 — a point on the unit circle

`angle_to_complex(θ)` maps a phase to `exp(iπθ)`: `θ = 0 → 1`, `θ = 0.5 → i`,
`θ = ±1 → -1`. Phases are points on the unit circle.

In [ ]:
thetas = Float32.(range(-1, 1, 9))
z = angle_to_complex(thetas)

circ = exp.(im .* range(0, 2pi, 200))
plot(real.(circ), imag.(circ), label="", color=:gray, aspect_ratio=:equal)
scatter!(real.(z), imag.(z), label="phases", xlabel="real", ylabel="imag",
         title="Face 1: phases as unit-circle points")

## Face 2 — a real angle

Stripped of the complex plane, a phase is just a number in `[-1, 1]`. This is the
formal FHRR symbol. `complex_to_angle` reads the angle back off the circle, so the
two faces are inverse views of the same value.

In [ ]:
recovered = complex_to_angle(z)
println("original θ : ", thetas)
println("recovered  : ", Float32.(recovered))

## Face 3 — a spike time

The third face is temporal: encode a phase as the *time* a neuron fires inside a
cycle of length `t_period`. Phase `-1` fires at the start of the cycle, `+1` at the
end. `phase_to_train` builds the spike train; `train_to_phase` reads it back.

In [ ]:
spk_args = SpikingArgs(t_period = 1.0f0)
ph = reshape(thetas, (:, 1))

train = phase_to_train(ph, spk_args=spk_args, repeats=1)
neuron = getindex.(train.indices, 1)      # row index of each spike
scatter(train.times, neuron, label="", xlabel="spike time (within cycle)",
        ylabel="neuron", title="Face 3: phases as spike times")

## Moving between the faces

The three faces are interchangeable. Here we round-trip a phase: angle → spike time → angle.

In [ ]:
t = phase_to_time(ph, spk_args=spk_args)          # angle  -> spike time
back = time_to_phase(vec(t), spk_args=spk_args, offset=0.0)   # time -> angle
println("angle in : ", vec(ph))
println("angle out: ", Float32.(vec(back)))

## Next

A phase is a static value with three faces. To **compute** with phases we give
them dynamics: an oscillator whose complex state rotates once per period and so
*carries* a phase through time. That is notebook 2.